## 5.3 Pytorch 模拟线性回归 - API实现

#### 1. 目标：
1. 用和上一节完全相同的数据，
2. 但这次使用：
    * nn.Linear
    * nn.MSELoss
    * torch.optim.SGD

#### 2. 先回顾：我们之前手写了什么？
1. 定义参数 w, b
2. 手动写 y_hat = w*x + b
3. 手动写 MSE
4. loss.backward()
5. w -= lr*w.grad
6. zero_()

现在我们用官方 API 替代这些步骤。


#### 3. 完整流程

##### 3.1 数据准备，延用上一小节的数据

In [5]:
import torch

x = torch.arange(1,51, dtype=torch.float32)
true_w = 3
true_b = 2
noise = torch.randn(x.shape) * 0.5
y = true_w * x + true_b + noise

x = x.reshape(-1,1)
y = y.reshape(-1,1)

##### 3.2 构建 模型，损失函数，优化器对象

In [6]:
import torch.nn as nn
# 1. 定义模型参数
model = nn.Linear(in_features=1, out_features=1) # 线性回归模型 y = wx + b
# 2. 定义损失函数和优化器
criterion = nn.MSELoss() # 均方误差损失函数
# 3. 定义优化器
optimizer = torch.optim.SGD(model.parameters(), lr=0.001) # 随机梯度下降优化器

##### 3.3 训练模型

In [7]:
epochs = 30
for epoch in range(1, epochs+1):
    # 1. forward
    y_pred = model(x)
    # 2. loss
    loss = criterion(y_pred, y)
    # 3. backward
    loss.backward()
    # 4. update parameters
    optimizer.step()
    # 5. zero gradients
    optimizer.zero_grad()
    if epoch % 5 == 0:        
        print(f'Epoch {epoch}, w = {model.weight.item():.4f}, b = {model.bias.item():.4f}, loss = {loss.item():.4f}')

Epoch 5, w = 3.6850, b = -0.5610, loss = 617.1699
Epoch 10, w = 2.9608, b = -0.5759, loss = 24.5914
Epoch 15, w = 3.0993, b = -0.5652, loss = 2.8507
Epoch 20, w = 3.0725, b = -0.5593, loss = 2.0449
Epoch 25, w = 3.0774, b = -0.5526, loss = 2.0070
Epoch 30, w = 3.0762, b = -0.5460, loss = 1.9972


#### 4. 流程详细拆解（非常重要）

##### 4.1 model = nn.Linear(1,1) 做了什么？
* 它内部自动创建：
    * `weight: shape (1,1)`
    * `bias: shape (1,)`
* 并且：
    * `weight.requires_grad = True`
    * `bias.requires_grad = True`
* 它帮你自动创建可训练参数。

##### 4.2 criterion = nn.MSELoss()
* 它帮你封装了：MSE 损失函数
* 不需要在 forward 中手动算损失函数 `((y_pred - y)**2).mean()`

##### 4.3 optimizer = optim.SGD(model.parameters())
* 这是最关键的一行
* model.parameters() 返回：
    * list(model.weight, model.bias)
* SGD 内部会：
    1. 读取参数的 .grad
    2. 按规则更新参数

##### 4.4 optimizer.step() 到底做了什么？

内部执行：
```
for param in model.parameters():
    param -= lr * param.grad
```
并且在 no_grad 环境中完成。


#### 4.5 optimizer.zero_grad() 和 手写的 zero_() 有什么区别？

之前我们写：
1. `w.grad.zero_()`
2. `b.grad.zero_()`

现在只需要：
* `optimizer.zero_grad()`
* 它会遍历 model.parameters()，全部清零